# License plate detection with YOLO26: data preparation and training

Dataset: [License Plate Recognition](https://www.kaggle.com/datasets/adilshamim8/license-plate-recognition) (Kaggle). Evaluation is in `02_evaluation.ipynb`.

In [ ]:
import os
import re
import glob
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

ROOT = '/kaggle/input/datasets/adilshamim8/license-plate-recognition'
WORK = '/kaggle/working'
SPLITS = ['train', 'valid', 'test']

## 1. Annotations

In [ ]:
frames = []
for split in SPLITS:
    folder = f'{ROOT}/{split}/{split}'
    df = pd.read_csv(f'{folder}/_annotations.csv')
    df['split'] = split
    df['folder'] = folder
    frames.append(df)
boxes = pd.concat(frames, ignore_index=True)

print(boxes.shape)
print(boxes['class'].value_counts())
boxes.head()

## 2. Image sources

In [ ]:
def source_of(filename):
    name = filename.split('.rf.')[0]
    name = re.sub(r'_(jpg|jpeg|png)$', '', name, flags=re.I)
    prefix = re.sub(r'[\d_\-\s]+$', '', name)
    return prefix or 'numbers_only'


per_image = boxes.drop_duplicates('filename').copy()
per_image['source'] = per_image['filename'].apply(source_of)

table = pd.crosstab(per_image['source'], per_image['split'], margins=True)
table.sort_values('All', ascending=False).head(15)

In [ ]:
for src in per_image['source'].value_counts().index[:4]:
    sample = per_image[(per_image['source'] == src) & (per_image['split'] == 'train')].sample(4, random_state=0)
    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    for ax, row in zip(axes, sample.itertuples()):
        ax.imshow(Image.open(f'{row.folder}/{row.filename}'))
        for b in boxes[boxes['filename'] == row.filename].itertuples():
            ax.add_patch(patches.Rectangle((b.xmin, b.ymin), b.xmax - b.xmin, b.ymax - b.ymin,
                                           lw=2, ec='lime', fc='none'))
        ax.axis('off')
    fig.suptitle(src)
    plt.show()

## 3. Image pool

The original Roboflow split is dropped: it puts augmented copies of the same images in both train and test.

In [ ]:
AUGMENTED = ['CarLongPlateGen', 'boderquandoi', 'brightnessquandoi',
             'rotatequandoi', 'cropquandoi']

rows = []
for split in SPLITS:
    for path in glob.glob(f'{ROOT}/{split}/{split}/*.jpg'):
        rows.append({'filename': os.path.basename(path), 'path': path})
images = pd.DataFrame(rows).sort_values('filename').reset_index(drop=True)

images['source'] = images['filename'].apply(source_of)
counts = images['source'].value_counts()
images['source_group'] = images['source'].where(images['source'].isin(counts[counts >= 50].index), 'other')
images['is_aug'] = images['source'].isin(AUGMENTED)
images['n_boxes'] = images['filename'].map(boxes.groupby('filename').size()).fillna(0).astype(int)

print(f'{len(images)} images, {images.is_aug.sum()} augmented copies, '
      f'{(images.n_boxes == 0).sum()} without any plate')
images['source_group'].value_counts()

## 4. Duplicate detection

Two dHash fingerprints per image: the whole image (64 bits) and the crop of the biggest plate (256 bits).

In [ ]:
def dhash(img, size):
    small = np.asarray(img.convert('L').resize((size + 1, size), Image.LANCZOS), dtype=np.int16)
    return (small[:, 1:] > small[:, :-1]).flatten()


boxes['area'] = (boxes.xmax - boxes.xmin) * (boxes.ymax - boxes.ymin)
main_box = boxes.sort_values('area', ascending=False).drop_duplicates('filename').set_index('filename')

n = len(images)
whole = np.zeros((n, 64), dtype=bool)
plate = np.zeros((n, 256), dtype=bool)

for i, row in enumerate(images.itertuples()):
    img = Image.open(row.path).convert('RGB')
    whole[i] = dhash(img, 8)

    crop = img
    if row.filename in main_box.index:
        b = main_box.loc[row.filename]
        c = img.crop((max(0, b.xmin), max(0, b.ymin), min(img.width, b.xmax), min(img.height, b.ymax)))
        if c.width >= 4 and c.height >= 4:
            crop = c
    plate[i] = dhash(crop, 16)

In [ ]:
def hamming(a, b):
    a, b = a.astype(np.float32), b.astype(np.float32)
    return a @ (1 - b).T + (1 - a) @ b.T


parent = list(range(n))

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra


PARKING = ['xemay', 'xemayBigPlate', 'CarLongPlate', 'CarLongPlateGen']
diverse = ~images['source_group'].isin(PARKING).to_numpy()

for s in range(0, n, 1000):
    e = min(s + 1000, n)
    dp = hamming(plate[s:e], plate)
    dw = hamming(whole[s:e], whole)
    k = np.arange(e - s)
    dp[k, k + s] = 999
    dw[k, k + s] = 999

    same_image = (dw <= 4) & (dp <= 60)
    same_car = (dp <= 24) & (dw <= 16)
    same_plate = diverse[s:e, None] & diverse[None, :] & (dp <= 32)

    for a, b in zip(*np.where(same_image | same_car | same_plate)):
        if a + s < b:
            union(a + s, b)

images['group'] = [find(i) for i in range(n)]
sizes = images['group'].value_counts()
print(f'{images.group.nunique()} groups, largest one has {sizes.max()} images')

## 5. Split

Parking-camera and augmented sources go to train only. The other photos are split by group, 60 / 20 / 20.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

TRAIN_ONLY = ['CarLongPlate', 'CarLongPlateGen', 'quandoi', 'boderquandoi', 'brightnessquandoi',
              'rotatequandoi', 'cropquandoi', 'xemay', 'xemayBigPlate']

images['train_only'] = images['source_group'].isin(TRAIN_ONLY)
images['group_train_only'] = images['group'].map(images.groupby('group')['train_only'].any())

cand = images[~images['group_train_only']]
fold = pd.Series(-1, index=cand.index)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
for k, (_, idx) in enumerate(sgkf.split(cand, cand['source_group'], cand['group'])):
    fold.iloc[idx] = k

group_split = dict(zip(cand['group'], fold.map({0: 'test', 1: 'valid'}).fillna('train')))
images['split'] = images['group'].map(group_split).fillna('train')

assert images.groupby('group')['split'].nunique().max() == 1
assert not (images['train_only'] & images['split'].isin(['valid', 'test'])).any()

images.to_csv(f'{WORK}/pool_split.csv', index=False)
pd.crosstab(images['source_group'], images['split'], margins=True)

## 6. YOLO dataset

In [ ]:
DST = f'{WORK}/yolo'
shutil.rmtree(DST, ignore_errors=True)
for split in SPLITS:
    os.makedirs(f'{DST}/{split}/images')
    os.makedirs(f'{DST}/{split}/labels')

boxes_by_file = dict(tuple(boxes.groupby('filename')))

for r in images.itertuples():
    shutil.copy(r.path, f'{DST}/{r.split}/images/{r.filename}')
    lines = []
    if r.filename in boxes_by_file:
        for b in boxes_by_file[r.filename].itertuples():
            x1, x2 = np.clip([b.xmin, b.xmax], 0, b.width)
            y1, y2 = np.clip([b.ymin, b.ymax], 0, b.height)
            if x2 - x1 < 2 or y2 - y1 < 2:
                continue
            lines.append(f'0 {(x1 + x2) / 2 / b.width:.6f} {(y1 + y2) / 2 / b.height:.6f} '
                         f'{(x2 - x1) / b.width:.6f} {(y2 - y1) / b.height:.6f}')
    with open(f'{DST}/{r.split}/labels/{r.filename[:-4]}.txt', 'w') as f:
        f.write('\n'.join(lines))

with open(f'{DST}/data.yaml', 'w') as f:
    f.write(f'path: {DST}\ntrain: train/images\nval: valid/images\ntest: test/images\n'
            'names:\n  0: License_Plate\n')

for split in SPLITS:
    print(split, len(os.listdir(f'{DST}/{split}/images')))

## 7. Training

`%%capture` hides the per-epoch log, which crashes the Kaggle tab on long runs.

In [ ]:
%%capture
!pip install -q -U ultralytics
from ultralytics import YOLO

model = YOLO('yolo26s.pt')
model.train(
    data=f'{WORK}/yolo/data.yaml',
    epochs=30, imgsz=640, batch=16, patience=15,
    project=f'{WORK}/runs', name='baseline_yolo26s',
    degrees=5, perspective=0.0005, hsv_v=0.5,
    workers=2, exist_ok=True,
)

In [ ]:
res = pd.read_csv(f'{WORK}/runs/baseline_yolo26s/results.csv')
res.columns = res.columns.str.strip()

metrics = {'metrics/precision(B)': 'precision', 'metrics/recall(B)': 'recall',
           'metrics/mAP50(B)': 'mAP50', 'metrics/mAP50-95(B)': 'mAP50-95'}
res[['epoch', *metrics]].rename(columns=metrics).tail(10).round(3)